In [ ]:
!pip3 install wordcloud

In [ ]:

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Load the categorized data into a DataFrame
df = pd.read_csv('data/stanza_annotations.csv')

# Inspect the data
print(df.head())


In [ ]:
for category in ["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"]:
    df[category] = df[category].apply(lambda x: 1 if x == 'o' else 0)

df_dict = df.to_dict(orient='records')
print(df_dict[:2])

In [ ]:
song_analysis={}

for x in df_dict:
    song_title = x['Song title']
    artist = x['Artist']
    year = x['Year']
    
    if song_title not in song_analysis:
        song_analysis[song_title] = {
            'Song artist' : artist,
            'Year' : year,
            'Total stanzas' : x['Total stanzas'],
            'Reinforce gender stereotype' : 0,
            'Challenge gender stereotype' : 0,
            'Self-awareness' : 0
        }
        
    song_analysis[song_title]['Reinforce gender stereotype'] += x['Reinforce gender stereotype']
    song_analysis[song_title]['Challenge gender stereotype'] += x['Challenge gender stereotype']
    song_analysis[song_title]['Self-awareness'] += x['Self-awareness']
    
print(song_analysis)

In [ ]:
for song, data in song_analysis.items():
    total_stanzas = data['Total stanzas']
    song_artist = data['Song artist']
    year = data['Year']
    data['Reinforce gender stereotype proportion'] = data['Reinforce gender stereotype'] / total_stanzas
    data['Challenge gender stereotype proportion'] = data['Challenge gender stereotype'] / total_stanzas
    data['Self-awareness proportion'] = data['Self-awareness'] / total_stanzas

for song, data in song_analysis.items():
    print(f"Song: {song}")
    print(f"  Year: {data['Year']}")
    print(f"  Song artist: {data['Song artist']}")
    print(f"  Total stanzas: {data['Total stanzas']}")
    print(f"  Reinforce gender stereotype: {data['Reinforce gender stereotype']} ({data['Reinforce gender stereotype proportion']:.2f})")
    print(f"  Challenge gender stereotype: {data['Challenge gender stereotype']} ({data['Challenge gender stereotype proportion']:.2f})")
    print(f"  Self-awareness: {data['Self-awareness']} ({data['Self-awareness proportion']:.2f})")
    print()
    
print()

In [ ]:
data_visualized = []
for song, data in song_analysis.items():
    data_visualized.append({
        'Song Title': song,
        'Artist': data['Song artist'],
        'Year': data['Year'],
        'Category': 'Reinforce gender stereotype',
        'Proportion': data['Reinforce gender stereotype proportion']
    })
    data_visualized.append({
        'Song Title': song,
        'Artist': data['Song artist'],
        'Year': data['Year'],
        'Category': 'Challenge gender stereotype',
        'Proportion': data['Challenge gender stereotype proportion']
    })
    data_visualized.append({
        'Song Title': song,
        'Artist': data['Song artist'],
        'Year': data['Year'],
        'Category': 'Self-awareness',
        'Proportion': data['Self-awareness proportion']
    })

# Convert to DataFrame for easy visualization
df_visualization = pd.DataFrame(data_visualized)
print(df_visualization)

In [ ]:
color_map = {
    'Reinforce gender stereotype': '#83c9f4',  # Blue
    'Challenge gender stereotype': '#fed766',  # Red
    'Self-awareness': '#b7e4c7'  # Green
}

fig_stacked = px.bar(df_visualization,
                     y='Song Title',
                     x='Proportion',
                     color='Category',
                     title='Proportion of Categories for Each Song',
                     labels={'Song Title': 'Song Title', 'Proportion': 'Proportion of Stanzas', 'Category': 'Category'},
                     barmode='stack',
                     color_discrete_map=color_map,
                     width=1000, height=1000)

fig_stacked.show()

In [ ]:
fig_grouped_song = px.bar(df_visualization,
                          y='Song Title',
                          x='Proportion',
                          color='Category',
                          title='Categories Comparison Across Songs',
                          labels={'Song Title': 'Song Title', 'Proportion': 'Category Proportion of Stanzas', 'Category': 'Category'},
                          barmode='group',
                          color_discrete_map=color_map,
                          width=1000, height=1500)

# Show the plot
fig_grouped_song.show()

In [ ]:
fig_grouped_artist = px.bar(df_visualization,
                            y='Artist',
                            x='Proportion',
                            color='Category',
                            title='Categories Comparison Across Artists',
                            labels={'Artist': 'Artist', 'Proportion': 'Category Proportion of Stanzas', 'Category': 'Category'},
                            barmode='group',
                            color_discrete_map=color_map,
                            width=1000, height=1500)

fig_grouped_artist.show()

In [ ]:
# Calculate the average proportion of each category across all songs
average_proportions = df_visualization.groupby('Category')['Proportion'].mean().reset_index()

print(average_proportions)

# Create a bar chart for average proportions
fig_average = px.bar(average_proportions,
                     x='Category',
                     y='Proportion',
                     title='Average Categories Proportion Across All Songs',
                     labels={'Category': 'Category', 'Proportion': 'Average Proportion'},
                     color='Category',
                     color_discrete_map=color_map,)

fig_average.show()

In [ ]:
# Calculate yearly proportions for each category across all songs
df_year_summary = df_visualization.groupby(['Year', 'Category'])['Proportion'].mean().reset_index()
print(df_year_summary)

# Create a line chart to visualize the category proportions over time
fig_line_year = px.line(df_year_summary,
                        x='Year',
                        y='Proportion',
                        color='Category',
                        color_discrete_map=color_map,
                        title='Category Proportions Over Time',
                        labels={'Year': 'Year', 'Proportion': 'Average Proportion of Stanzas', 'Category': 'Category'},
                        markers=True)

fig_line_year.show()

In [ ]:
# Filter data for a specific artist (e.g., "Donna Summer")
artist_name = "Donna Summer"
df_artist = df_visualization[df_visualization['Artist'] == artist_name]

# Create a pie chart for the selected artist
fig_artist_pie = px.pie(df_artist,
                        names='Category',
                        color='Category',
                        color_discrete_map=color_map,
                        values='Proportion',
                        title=f'Category Breakdown for {artist_name}')

fig_artist_pie.show()

In [ ]:
# Group by song title to get counts of each category
category_counts = df.groupby('Song title')[["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"]].sum().reset_index()

# Add total stanza count for each song
category_counts = category_counts.merge(df[['Song title', 'Total stanzas']].drop_duplicates(), on='Song title')

# View the category counts DataFrame
print(category_counts)

In [ ]:
# Melt the DataFrame to prepare for visualization
category_counts_melted = category_counts.melt(id_vars=['Song title', 'Total stanzas'],
                                              value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                                              var_name='Category', value_name='Count')

# Create a stacked bar chart using Plotly
fig_stacked_count = px.bar(category_counts_melted,
                           y='Song title',
                           x='Count',
                           color='Category',
                           title='Count of Stanzas in Each Category for Each Song',
                           labels={'song title': 'Song Title', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                           barmode='stack',
                           color_discrete_map=color_map,
                           width=1000, height=1500
                           )

fig_stacked_count.show()

In [ ]:
# Melt the DataFrame to prepare for visualization
category_counts_melted = category_counts.melt(id_vars=['Song title', 'Total stanzas'],
                                              value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                                              var_name='Category', value_name='Count')

# Create a stacked bar chart using Plotly
fig_stacked_count = px.bar(category_counts_melted,
                           y='Song title',
                           x='Count',
                           color='Category',
                           title='Count of Stanzas in Each Category for Each Song',
                           labels={'song title': 'Song Title', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                           barmode='group',
                           color_discrete_map=color_map,
                           width=1000, height=1500
                           )

fig_stacked_count.show()

In [ ]:
# Create new columns for combinations of categories
df['Reinforce & Challenge'] = df['Reinforce gender stereotype'] & df['Challenge gender stereotype']
df['Reinforce & Self-Awareness'] = df['Reinforce gender stereotype'] & df['Self-awareness']
df['Challenge & Self-Awareness'] = df['Challenge gender stereotype'] & df['Self-awareness']
df['All Three'] = df['Reinforce gender stereotype'] & df['Challenge gender stereotype'] & df['Self-awareness']

# Group by song title to get counts for each overlap
overlap_counts = df.groupby('Song title')[["Reinforce & Challenge", "Reinforce & Self-Awareness", "Challenge & Self-Awareness", "All Three"]].sum().reset_index()

# View overlap counts
print(overlap_counts)


In [ ]:
# 確保已經有數據框包含每首歌曲的段落數量
df['Total Stanzas'] = df.groupby('Song title')['Stanza Number'].transform('count')

# 使用重疊組合的總數量來計算比例
overlap_counts = df.groupby('Song title')[["Reinforce & Challenge", "Reinforce & Self-Awareness", "Challenge & Self-Awareness", "All Three"]].sum().reset_index()

# 將每個重疊組合的比例計算出來
for column in ["Reinforce & Challenge", "Reinforce & Self-Awareness", "Challenge & Self-Awareness", "All Three"]:
    overlap_counts[column + ' Proportion'] = overlap_counts[column] / df['Total Stanzas']
    
# # 查看計算出的重疊比例
# print(overlap_counts)

# 計算每個重疊組合在所有歌曲中的平均比例
average_proportions = overlap_counts[["Reinforce & Challenge Proportion", "Reinforce & Self-Awareness Proportion", "Challenge & Self-Awareness Proportion", "All Three Proportion"]].mean().reset_index()
average_proportions.columns = ['Overlap Category', 'Average Proportion']

# 查看平均比例
print(average_proportions)

import plotly.express as px

overlap_color_map = {
    'Reinforce & Challenge Proportion': '#ff7f11',       # orange
    'Reinforce & Self-Awareness Proportion': '#219ebc',  # greenblue
    'Challenge & Self-Awareness Proportion': '#ff90b3',  # Pink
    'All Three Proportion': '#b892ff'                    # Purple
}

# 使用 plotly 創建重疊組合的平均比例圖
fig = px.bar(average_proportions,
             x='Overlap Category',
             y='Average Proportion',
             title='Average Proportion of Overlap Combinations Across All Songs',
             labels={'Overlap Category': 'Overlap Combination', 'Average Proportion': 'Average Proportion'},
             color='Overlap Category',
             color_discrete_map=overlap_color_map,)

# 顯示圖表
fig.show()


In [ ]:
# Melt the overlap counts DataFrame to prepare for visualization
overlap_counts_melted = overlap_counts.melt(id_vars=['Song title'],
                                            value_vars=["Reinforce & Challenge", "Reinforce & Self-Awareness", "Challenge & Self-Awareness", "All Three"],
                                            var_name='Overlap Category', value_name='Count')

overlap_color_map = {
    'Reinforce & Challenge': '#ff7f11',       # orange
    'Reinforce & Self-Awareness': '#219ebc',  # greenblue
    'Challenge & Self-Awareness': '#ff90b3',  # Pink
    'All Three': '#b892ff'                    # Purple
}


# Create a stacked bar chart for overlaps
fig_overlap_stacked = px.bar(overlap_counts_melted,
                             y='Song title',
                             x='Count',
                             color='Overlap Category',
                             title='Count of Stanzas Falling into Multiple Categories',
                             labels={'song title': 'Song Title', 'Count': 'Number of Stanzas', 'Overlap Category': 'Overlap Combination'},
                             barmode='group',
                             color_discrete_map=overlap_color_map,
                             width=1000, height=1000)

fig_overlap_stacked.show()


In [ ]:
# Melt the overlap counts DataFrame to prepare for visualization
overlap_counts_melted = overlap_counts.melt(id_vars=['Song title'],
                                            value_vars=["Reinforce & Challenge", "Reinforce & Self-Awareness", "Challenge & Self-Awareness", "All Three"],
                                            var_name='Overlap Category', value_name='Count')

overlap_color_map = {
    'Reinforce & Challenge': '#ff7f11',       # orange
    'Reinforce & Self-Awareness': '#219ebc',  # greenblue
    'Challenge & Self-Awareness': '#ff90b3',  # Pink
    'All Three': '#b892ff'                    # Purple
}


# Create a stacked bar chart for overlaps
fig_overlap_stacked = px.bar(overlap_counts_melted,
                             y='Song title',
                             x='Count',
                             color='Overlap Category',
                             title='Count of Stanzas Falling into Multiple Categories',
                             labels={'song title': 'Song Title', 'Count': 'Number of Stanzas', 'Overlap Category': 'Overlap Combination'},
                             barmode='stack',
                             color_discrete_map=overlap_color_map,
                             width=1000, height=1500)

fig_overlap_stacked.show()

In [ ]:
pip install matplotlib-venn

In [ ]:
from matplotlib_venn import venn3
import matplotlib.pyplot as plt

# Sum the overlap counts to get total counts across all songs
reinforce_challenge = overlap_counts['Reinforce & Challenge'].sum()
reinforce_self = overlap_counts['Reinforce & Self-Awareness'].sum()
challenge_self = overlap_counts['Challenge & Self-Awareness'].sum()
all_three = overlap_counts['All Three'].sum()

# Plot a Venn diagram for the overlapping categories
plt.figure(figsize=(20, 15))
venn3(subsets=(reinforce_challenge, reinforce_self, challenge_self, all_three, 0, 0, 0),
      set_labels=('Reinforce Gender Stereotype', 'Challenge Gender Stereotype', 'Self-Awareness'))
plt.title("Venn Diagram of Overlapping Categories")
plt.show()


In [ ]:
import numpy as np

# Create a correlation DataFrame to track category overlaps
overlap_corr = df[['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness']].corr()

print(overlap_corr)

In [ ]:
import plotly.graph_objects as go

# Create a heatmap for the correlation of categories
fig_heatmap = go.Figure(data=go.Heatmap(
    z=overlap_corr.values,
    x=['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness'],
    y=['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness'],
    colorscale='Viridis'))

# Update the layout of the heatmap
fig_heatmap.update_layout(
    title='Heatmap of Overlapping Categories',
    xaxis_title='Category',
    yaxis_title='Category'
)

fig_heatmap.show()


In [ ]:
# Ensure that year is in integer format
df['Year'] = df['Year'].astype(int)

# Melt the DataFrame to create separate rows for each category
df_melted = df.melt(id_vars=['Song title', 'Artist', 'Stanza Number', 'Stanza Text', 'Year', 'Total stanzas'],
                    value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                    var_name='Category', value_name='CategoryValue')

# Filter out rows where the category value is 0 (meaning the stanza doesn't belong to that category)
df_melted = df_melted[df_melted['CategoryValue'] == 1]

# Group by year and category to get the count of stanzas in each category for each year
df_year_summary = df_melted.groupby(['Year', 'Category']).size().reset_index(name='Count')

# # Display the DataFrame to verify
# print(df_year_summary.head())


# Create a line chart to show category trends over time
fig_line_trends = px.line(df_year_summary,
                          x='Year',
                          y='Count',
                          color='Category',
                          title='Category Trends Over Time',
                          labels={'year': 'Year', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                          markers=True,
                          color_discrete_map=color_map)  # Use the consistent color map

fig_line_trends.update_traces(line_shape='linear')  # Set line shape to 'linear' for smoother lines
fig_line_trends.show()

In [ ]:
# Drop duplicates to avoid aggregating incorrectly
df_unique_stanzas = df[['Song title', 'Artist', 'Total stanzas']].drop_duplicates()

# Add a column for 'Artist_Song' to be used in the sunburst visualization
df_unique_stanzas['Artist_Song'] = df_unique_stanzas['Artist'] + " - " + df_unique_stanzas['Song title']

# # Display the DataFrame to verify
# print(df_unique_stanzas.head())

# Create the sunburst chart using the cleaned DataFrame
fig_sunburst = px.sunburst(df_unique_stanzas,
                           path=['Artist', 'Song title', 'Artist_Song'],  # Hierarchical path for the sunburst
                           values='Total stanzas',                         # Correct total stanzas value for each song
                           color='Artist',                                 # Color by artist
                           title='Hierarchy of Artists, Songs, and Stanzas',
                           width=800,
                           height=800)

# Update layout for better visualization
fig_sunburst.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    title_font_size=20,
    uniformtext=dict(minsize=10, mode='hide'),
    colorway=["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]  # Custom color palette for differentiation
)

# Show the sunburst chart
fig_sunburst.show()

In [ ]:
# Calculate the sum for each category by year
year_summary = df.groupby('Year')[["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"]].sum().reset_index()

# Melt the DataFrame to prepare for visualization
df_year_grouped = year_summary.melt(id_vars=['Year'], 
                                    value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                                    var_name='Category', value_name='Count')

# Create a grouped bar chart by year to compare different categories
fig_grouped_year = px.bar(df_year_grouped,
                          x='Year',
                          y='Count',
                          color='Category',
                          title='Category Comparison by Year',
                          labels={'year': 'Year', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                          barmode='group',
                          width = 1000,
                          color_discrete_map=color_map)  # Apply consistent color

fig_grouped_year.show()

In [ ]:
# Create a summary DataFrame for each category
category_bubble_summary = df.groupby(['Song title', 'Artist']).agg({'Reinforce gender stereotype': 'sum',
                                                                    'Challenge gender stereotype': 'sum',
                                                                    'Self-awareness': 'sum',
                                                                    'Total stanzas': 'max'}).reset_index()

# Melt the DataFrame for bubble chart visualization
bubble_summary_melted = category_bubble_summary.melt(id_vars=['Song title', 'Artist', 'Total stanzas'],
                                                     value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                                                     var_name='Category', value_name='Count')

# Filter out rows where Count is 0 to avoid showing bubbles with zero values
bubble_summary_filtered = bubble_summary_melted[bubble_summary_melted['Count'] > 0]

# Create a bubble chart using Plotly
fig_bubble = px.scatter(bubble_summary_filtered,
                        y='Song title',
                        x='Count',
                        size='Total stanzas',
                        color='Category',
                        hover_name='Artist',
                        title='Stanza Distribution by Category and Song',
                        labels={'song title': 'Song Title', 'Count': 'Number of Stanzas', 'total stanzas': 'Total Stanzas', 'Category': 'Category'},
                        color_discrete_map=color_map,
                        width=1000, height=1200)  # Apply consistent color

# Set x-axis to start from 0
fig_bubble.update_xaxes(range=[0, None])

fig_bubble.show()


In [ ]:
# Create a treemap to visualize the hierarchical relationships
fig_treemap = px.treemap(df_unique_stanzas,
                         path=['Artist', 'Song title', 'Artist_Song'],  # Hierarchical path for the treemap
                         values='Total stanzas',                          # Values determine the size of rectangles
                         color='Artist',                                  # Color by artist
                         title='Treemap of Artists, Songs, and Stanzas',
                         width=900,
                         height=700,
                         color_discrete_map=color_map)  # Using the consistent color map

# Update layout for better visualization
fig_treemap.update_layout(
    margin=dict(t=50, l=25, r=25, b=25),
    title_font_size=20
)

# Show the treemap
fig_treemap.show()


In [ ]:
# Create a histogram for the number of stanzas per song
fig_histogram = px.histogram(df_unique_stanzas,
                             x='Total stanzas',
                             nbins=20,
                             title='Distribution of Total Stanzas per Song',
                             labels={'total stanzas': 'Total Stanzas'},
                             color_discrete_sequence=['#636EFA'])  # Use consistent color

# Update layout for better appearance
fig_histogram.update_layout(
    width=800,
    height=600,
    title_font_size=20,
    margin=dict(t=50, l=25, r=25, b=25)
)

# Show the histogram
fig_histogram.show()


In [ ]:
import plotly.graph_objects as go

# Melt the DataFrame to create a categorical column named 'Category'
df_melted = df.melt(id_vars=['Song title', 'Artist', 'Stanza Number', 'Stanza Text', 'Year', 'Total stanzas'],
                    value_vars=["Reinforce gender stereotype", "Challenge gender stereotype", "Self-awareness"],
                    var_name='Category', value_name='CategoryValue')

# Keep only rows where the CategoryValue is 1 (indicating the stanza falls into the category)
df_melted = df_melted[df_melted['CategoryValue'] == 1]

# Group by year and category to get the count of stanzas in each category for each year
heatmap_data = df_melted.groupby(['Year', 'Category']).size().unstack().fillna(0).astype(int)

# Display heatmap data to verify
print(heatmap_data.head())

import plotly.graph_objects as go

# Create a heatmap using Plotly
fig_heatmap = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,          # The values for the heatmap
    x=heatmap_data.columns,         # The categories on the x-axis
    y=heatmap_data.index,           # The years on the y-axis
    colorscale='Viridis'            # Colorscale for the heatmap
))

# Update layout for better visualization
fig_heatmap.update_layout(
    title='Heatmap of Category Occurrence Over the Years',
    xaxis_title='Category',
    yaxis_title='Year',
    width=900,
    height=900,
    title_font_size=20
)

# Show the heatmap
fig_heatmap.show()


In [ ]:
# Prepare data for stacked area chart
# Group by year and category, summing the counts for each category per year
df_area_summary = df_melted.groupby(['Year', 'Category']).size().reset_index(name='Count')

# Create a stacked area chart using Plotly Express
fig_area = px.area(df_area_summary,
                   x='Year',
                   y='Count',
                   color='Category',
                   title='Stacked Area Chart of Categories Over Time',
                   labels={'year': 'Year', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                   color_discrete_map=color_map)  # Use the consistent color map

# Update layout for better appearance
fig_area.update_layout(
    width=900,
    height=700,
    title_font_size=20,
    margin=dict(t=50, l=25, r=25, b=25)
)

# Show the stacked area chart
fig_area.show()


In [ ]:
# Group by 'Category' to get the total count of stanzas for each category
category_counts = df_melted['Category'].value_counts().reset_index()
category_counts.columns = ['Category', 'Count']

# Display the DataFrame to verify
print(category_counts)

# Create a bar chart to visualize the counts of each category
fig_category_counts = px.bar(category_counts,
                             x='Category',
                             y='Count',
                             title='Counts of Each Category Across All Stanzas',
                             labels={'Category': 'Category', 'Count': 'Number of Stanzas'},
                             color='Category',
                             color_discrete_map=color_map)

# Update layout for better appearance
fig_category_counts.update_layout(
    width=800,
    height=600,
    title_font_size=20,
    margin=dict(t=50, l=25, r=25, b=25)
)

# Show the bar chart
fig_category_counts.show()

In [ ]:
# Group by 'artist' and 'Category' to get the total count of stanzas for each category by artist
artist_category_counts = df_melted.groupby(['Artist', 'Category']).size().reset_index(name='Count')

# Display the DataFrame to verify
print(artist_category_counts)

# Create a grouped bar chart to visualize the counts of each category by artist
fig_artist_category = px.bar(artist_category_counts,
                             x='Artist',
                             y='Count',
                             color='Category',
                             title='Distribution of Categories by Artist',
                             labels={'artist': 'Artist', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                             barmode='group',  # Grouped bar chart to show counts for each category side-by-side
                             color_discrete_map= color_map)

# Update layout for better appearance
fig_artist_category.update_layout(
    width=1100,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,  # Rotate artist labels to make them more readable
    margin=dict(t=50, l=25, r=25, b=75)
)

# Show the grouped bar chart
fig_artist_category.show()

In [ ]:
# Group by 'Category' and 'artist' to get the total count of stanzas for each category by artist
category_artist_counts = df_melted.groupby(['Category', 'Artist']).size().reset_index(name='Count')

# Display the DataFrame to verify
print(category_artist_counts)

# Create a stacked bar chart to visualize the counts of each artist by category
fig_category_artist_stacked = px.bar(category_artist_counts,
                                     x='Category',
                                     y='Count',
                                     color='Artist',
                                     title='Distribution of Stanzas by Category and Artist',
                                     labels={'Category': 'Category', 'Count': 'Number of Stanzas', 'artist': 'Artist'},
                                     barmode='stack',  # Stacked to show artist contributions within each category
                                     width=1000,
                                     height=600)

# Update layout for better appearance
fig_category_artist_stacked.update_layout(
    title_font_size=20,
    xaxis_tickangle=-45,
    margin=dict(t=50, l=25, r=25, b=75)
)

# Show the stacked bar chart
fig_category_artist_stacked.show()



In [ ]:
# Create a faceted bar chart to visualize artist contribution for each category separately
fig_category_artist_facet = px.bar(category_artist_counts,
                                   x='Artist',
                                   y='Count',
                                   color='Artist',
                                   title='Artist Contribution by Category',
                                   labels={'artist': 'Artist', 'Count': 'Number of Stanzas', 'Category': 'Category'},
                                   facet_col='Category',  # Create separate charts for each category
                                   width=1000,
                                   height=600)

# Update layout for better appearance
fig_category_artist_facet.update_layout(
    title_font_size=20,
    xaxis_tickangle=-45,
    margin=dict(t=50, l=25, r=25, b=75)
)

# Show the faceted bar chart
fig_category_artist_facet.show()

In [ ]:
# Filter to keep only the specific category 'Challenge gender stereotype'
category_filtered = df_melted[df_melted['Category'] == 'Challenge gender stereotype']

# Group by 'artist' to get the count of stanzas for this specific category
artist_counts = category_filtered['Artist'].value_counts().reset_index()
artist_counts.columns = ['Artist', 'Count']

# Calculate the total count for proportions
total_count = artist_counts['Count'].sum()

# Add a new column for proportion
artist_counts['Proportion'] = artist_counts['Count'] / total_count

# Display the DataFrame to verify
print(artist_counts)

# Create a bar chart to visualize the proportion of each artist for the specific category
fig_proportion_bar = px.bar(artist_counts,
                            x='Artist',
                            y='Proportion',
                            title='Proportion of stanzas fall into "Challenge Gender Stereotype" Category sorted by Artist',
                            labels={'Artist': 'Artist', 'Proportion': 'Proportion of Stanzas'},
                            text='Proportion',  # Display proportion values on bars
                            color='Artist')

# Update layout for better appearance
fig_proportion_bar.update_layout(
    width=900,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_proportion_bar.show()

In [ ]:
# Filter to keep only the specific category 'Challenge gender stereotype'
category_filtered = df_melted[df_melted['Category'] == 'Reinforce gender stereotype']

# Group by 'artist' to get the count of stanzas for this specific category
artist_counts = category_filtered['Artist'].value_counts().reset_index()
artist_counts.columns = ['Artist', 'Count']

# Calculate the total count for proportions
total_count = artist_counts['Count'].sum()

# Add a new column for proportion
artist_counts['Proportion'] = artist_counts['Count'] / total_count

# Display the DataFrame to verify
print(artist_counts)

# Create a bar chart to visualize the proportion of each artist for the specific category
fig_proportion_bar = px.bar(artist_counts,
                            x='Artist',
                            y='Proportion',
                            title='Proportion of stanzas fall into "Reinforce Gender Stereotype" Category sorted by Artist',
                            labels={'Artist': 'Artist', 'Proportion': 'Proportion of Stanzas'},
                            text='Proportion',  # Display proportion values on bars
                            color='Artist')

# Update layout for better appearance
fig_proportion_bar.update_layout(
    width=900,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_proportion_bar.show()

In [ ]:
# Filter to keep only the specific category 'Challenge gender stereotype'
category_filtered = df_melted[df_melted['Category'] == 'Self-awareness']

# Group by 'artist' to get the count of stanzas for this specific category
artist_counts = category_filtered['Artist'].value_counts().reset_index()
artist_counts.columns = ['Artist', 'Count']

# Calculate the total count for proportions
total_count = artist_counts['Count'].sum()

# Add a new column for proportion
artist_counts['Proportion'] = artist_counts['Count'] / total_count

# Display the DataFrame to verify
print(artist_counts)

# Create a bar chart to visualize the proportion of each artist for the specific category
fig_proportion_bar = px.bar(artist_counts,
                            x='Artist',
                            y='Proportion',
                            title='Proportion of stanzas fall into "Self-awareness" Category sorted by Artist',
                            labels={'Artist': 'Artist', 'Proportion': 'Proportion of Stanzas'},
                            text='Proportion',  # Display proportion values on bars
                            color='Artist')

# Update layout for better appearance
fig_proportion_bar.update_layout(
    width=900,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_proportion_bar.show()

In [ ]:
# Keep only rows where the Category is 'Self-awareness' and CategoryValue is 1
self_awareness_data = df_melted[(df_melted['Category'] == 'Self-awareness') & (df_melted['CategoryValue'] == 1)]

# Group by 'song title' to get the count of stanzas in the "Self-awareness" category for each song
self_awareness_counts = self_awareness_data.groupby('Song title').size().reset_index(name='Self-awareness Count')

# Calculate the total number of stanzas categorized as 'Self-awareness' across all songs
total_self_awareness_stanzas = self_awareness_counts['Self-awareness Count'].sum()

# Calculate the proportion of stanzas for each song relative to the total stanzas categorized as 'Self-awareness'
self_awareness_counts['Proportion'] = self_awareness_counts['Self-awareness Count'] / total_self_awareness_stanzas

# Sort by proportion in descending order
proportion_data_sorted = self_awareness_counts.sort_values(by='Proportion', ascending=False)

# # Display the DataFrame to verify
# print(proportion_data_sorted)

# Create a bar chart to visualize the proportion of "Self-awareness" stanzas by each song
fig_self_awareness = px.bar(proportion_data_sorted,
                            x='Song title',
                            y='Proportion',
                            title='Proportion of Stanzas fall into "Self-awareness" Category sorted by Song',
                            labels={'song title': 'Song Title', 'Proportion': 'Proportion of Self-awareness Stanzas'},
                            # text='Proportion',  # Display proportion values on bars
                            color='Proportion',
                            color_continuous_scale='greens')  # Use a color scale to represent proportions

# Update layout for better appearance
fig_self_awareness.update_layout(
    width=1000,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,  # Rotate song labels to make them more readable
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_self_awareness.show()

In [ ]:
# Keep only rows where the Category is 'Reinforce Gender Stereotype' and CategoryValue is 1
reinforce_data = df_melted[(df_melted['Category'] == 'Reinforce gender stereotype') & (df_melted['CategoryValue'] == 1)]

# Group by 'song title' to get the count of stanzas in the "Reinforce Gender Stereotype" category for each song
reinforce_counts = reinforce_data.groupby('Song title').size().reset_index(name='Reinforce Count')

# Calculate the total number of stanzas categorized as 'Reinforce Gender Stereotype' across all songs
total_reinforce_stanzas = reinforce_counts['Reinforce Count'].sum()

# Calculate the proportion of stanzas for each song relative to the total stanzas categorized as 'Reinforce Gender Stereotype'
reinforce_counts['Proportion'] = reinforce_counts['Reinforce Count'] / total_reinforce_stanzas

# Sort by proportion in descending order
reinforce_data_sorted = reinforce_counts.sort_values(by='Proportion', ascending=False)

# Display the DataFrame to verify
print(reinforce_data_sorted)

# Create a bar chart to visualize the proportion of "Reinforce Gender Stereotype" stanzas by each song
fig_reinforce = px.bar(reinforce_data_sorted,
                       x='Song title',
                       y='Proportion',
                       title='Proportion of Stanzas fall into "Reinforce Gender Stereotype" Category sorted by Song',
                       labels={'song title': 'Song Title', 'Proportion': 'Proportion of Reinforce Stanzas'},
                       text='Proportion',  # Display proportion values on bars
                       color='Proportion',
                       color_continuous_scale='blues')  # Use a color scale to represent proportions

# Update layout for better appearance
fig_reinforce.update_layout(
    width=1000,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,  # Rotate song labels to make them more readable
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_reinforce.show()

In [ ]:
# Keep only rows where the Category is 'Challenge Gender Stereotype' and CategoryValue is 1
challenge_data = df_melted[(df_melted['Category'] == 'Challenge gender stereotype') & (df_melted['CategoryValue'] == 1)]

# Group by 'song title' to get the count of stanzas in the "Challenge Gender Stereotype" category for each song
challenge_counts = challenge_data.groupby('Song title').size().reset_index(name='Challenge Count')

# Calculate the total number of stanzas categorized as 'Challenge Gender Stereotype' across all songs
total_challenge_stanzas = challenge_counts['Challenge Count'].sum()

# Calculate the proportion of stanzas for each song relative to the total stanzas categorized as 'Challenge Gender Stereotype'
challenge_counts['Proportion'] = challenge_counts['Challenge Count'] / total_challenge_stanzas

# Sort by proportion in descending order
challenge_data_sorted = challenge_counts.sort_values(by='Proportion', ascending=False)

# Display the DataFrame to verify
print(challenge_data_sorted)

# Create a bar chart to visualize the proportion of "Challenge Gender Stereotype" stanzas by each song
fig_challenge = px.bar(challenge_data_sorted,
                       x='Song title',
                       y='Proportion',
                       title='Proportion of Stanzas fall into "Challenge Gender Stereotype" Category sorted by Song',
                       labels={'song title': 'Song Title', 'Proportion': 'Proportion of Challenge Stanzas'},
                       text='Proportion',  # Display proportion values on bars
                       color='Proportion',
                       color_continuous_scale='Reds')  # Use a color scale to represent proportions

# Update layout for better appearance
fig_challenge.update_layout(
    width=1000,
    height=600,
    title_font_size=20,
    xaxis_tickangle=-45,  # Rotate song labels to make them more readable
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_challenge.show()

In [ ]:
# Group by 'song title' and 'Category' to get the count of stanzas for each category in each song
category_counts = df_melted.groupby(['Song title', 'Category']).size().reset_index(name='Category Count')

# Calculate the total number of categorized stanzas per song by grouping only by 'song title'
total_categorized_stanzas = category_counts.groupby('Song title')['Category Count'].sum().reset_index(name='Total Categorized Stanzas')

# Merge the category counts with the total categorized stanzas for each song
proportion_data = pd.merge(category_counts, total_categorized_stanzas, on='Song title')

# Calculate the normalized proportion of stanzas for each category within each song
proportion_data['Normalized Proportion'] = proportion_data['Category Count'] / proportion_data['Total Categorized Stanzas']

# Display the DataFrame to verify
print(proportion_data)

# Create a normalized stacked bar chart to visualize the relative emphasis of each category in each song
fig_normalized_stacked = px.bar(proportion_data,
                                y='Song title',
                                x='Normalized Proportion',
                                color='Category',
                                title='Normalized Contribution of Each Category in Each Song',
                                labels={'song title': 'Song Title', 'Normalized Proportion': 'Proportion of Categorized Stanzas', 'Category': 'Category'},
                                barmode='group',  # Stacked bar chart to show proportions as part of a whole
                                width=1000,
                                height=1500,
                                color_discrete_map= color_map)

# Update layout for better appearance
fig_normalized_stacked.update_layout(
    title_font_size=20,
    yaxis_tickangle=-45,  # Rotate song labels for readability
    margin=dict(t=50, l=25, r=25, b=75)
)

fig_normalized_stacked.show()

In [ ]:
# color_map = {
#     'Reinforce gender stereotype': '#83c9f4',  # Blue
#     'Challenge gender stereotype': '#fed766',  # Red
#     'Self-awareness': '#b7e4c7'  # Green
# }

# # Create pie charts for multiple songs
# for song_title in proportion_data['Song title'].unique():
#     # Filter data for each song
#     song_data = proportion_data[proportion_data['Song title'] == song_title]
    
#     # Create and show the pie chart
#     fig_pie = px.pie(song_data,
#                      names='Category',
#                      values='Normalized Proportion',
#                      title=f'Proportion of Each Category in "{song_title}"',
#                      labels={'Category': 'Category', 'Normalized Proportion': 'Proportion of Categorized Stanzas'},
#                      color='Category',
#                      width=600,
#                      height=600,
#                      color_discrete_map= color_map)
#     fig_pie.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get the unique song titles to determine how many pie charts are needed
unique_songs = proportion_data['Song title'].unique()

# Create a subplot figure for all pie charts
fig = make_subplots(
    rows=len(unique_songs) // 5 + 1,  # Increase the number of columns, which reduces the number of rows
    cols=5,  # Use 5 columns to reduce the number of rows to make the plot more compact
    subplot_titles=[
        f'{song_title}<br>by {proportion_data[proportion_data["Song title"] == song_title]["Artist"].iloc[0]}'
        for song_title in unique_songs
    ],
    specs=[[{'type': 'pie'}] * 5] * (len(unique_songs) // 5 + 1),  # Specify that each plot type is a pie chart
    horizontal_spacing=0.05,  # Decrease horizontal space between subplots to make the figure more compact
    vertical_spacing=0.05   # Adjust vertical spacing to stay within the limit based on the number of rows
)

# Step 3: Add Pie Charts for Each Song
for index, song_title in enumerate(unique_songs):
    # Filter data for the selected song
    song_data = proportion_data[proportion_data['Song title'] == song_title]
    artist_name = song_data['Artist'].iloc[0]  # Get the artist name for the song

    # Get the row and column position for the current pie chart
    row = index // 5 + 1
    col = index % 5 + 1
    
    # Add pie chart to the subplot with the song title and artist name
    fig.add_trace(
        go.Pie(
            values=song_data['Normalized Proportion'],
            name=f"{song_title}<br>{artist_name}",  # Song title on the first line, artist on the second line
            marker=dict(colors=[color_map[cat] for cat in song_data['Category']]),
            hole=0.3,  # Make the pie chart slightly larger by reducing the hole size
            # textinfo='label+percent',  # Display category labels and percentages
            # textposition='inside'  # Place labels inside the pie slices
        ), 
        row=row, 
        col=col
    )

# Update layout for the entire figure
fig.update_layout(
    title=dict(
        text="Normalized Contribution of Each Category in Each Song",
        font=dict(size=20),
        y=0.98  # Adjust the title position to be higher, closer to the top of the figure
    ),
    height=300 * ((len(unique_songs) // 5) + 1),  # Adjust height to create more space between rows while reducing the number of rows
    width=1000,  # Increase the width of the entire figure to give more space for each subplot
    margin=dict(t=150, b=50),  # Increase top margin to prevent overlapping and create a gap between the title and subplots
    showlegend=False
)

# Reduce font size for subplot titles to reduce the chance of overlapping
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=12)  # Set font size of subplot titles

# Show the entire facet pie chart figure
fig.show()

In [ ]:
# Group by 'artist' and 'Category' to get the count of stanzas for each category for each artist
category_counts_artist = df_melted.groupby(['Artist', 'Category']).size().reset_index(name='Category Count')

# Calculate the total number of categorized stanzas per artist
total_categorized_stanzas_artist = category_counts_artist.groupby('Artist')['Category Count'].sum().reset_index(name='Total Categorized Stanzas')

# Merge the category counts with the total categorized stanzas for each artist
proportion_data_artist = pd.merge(category_counts_artist, total_categorized_stanzas_artist, on='Artist')

# Calculate the normalized proportion of stanzas for each category within each artist
proportion_data_artist['Normalized Proportion'] = proportion_data_artist['Category Count'] / proportion_data_artist['Total Categorized Stanzas']

# Display the DataFrame to verify
print(proportion_data_artist)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get the unique artist names to determine how many pie charts are needed
unique_artists = proportion_data_artist['Artist'].unique()

# Create a subplot figure for all pie charts
fig = make_subplots(
    rows=len(unique_artists) // 5 + 1,  # Calculate number of rows needed
    cols=5,  # Use 5 columns to create a balanced layout
    subplot_titles=[
        f'{artist}' for artist in unique_artists  # Titles for each artist
    ],
    specs=[[{'type': 'pie'}] * 5] * (len(unique_artists) // 5 + 1),  # Specify that each plot type is a pie chart
    horizontal_spacing=0.05,  # Adjust horizontal space between subplots to make the figure more compact
    vertical_spacing=0.1  # Adjust vertical space between subplots
)

# Step 3: Add Pie Charts for Each Artist
for index, artist in enumerate(unique_artists):
    # Filter data for the selected artist
    artist_data = proportion_data_artist[proportion_data_artist['Artist'] == artist]

    # Get the row and column position for the current pie chart
    row = index // 5 + 1
    col = index % 5 + 1
    
    # Add pie chart to the subplot with artist name
    fig.add_trace(
        go.Pie(
            # labels=artist_data['Category'],
            values=artist_data['Normalized Proportion'],
            name=f"{artist}",  # Artist name as the label
            marker=dict(colors=[color_map[cat] for cat in artist_data['Category']]),
            hole=0.3,  # Make the pie chart slightly larger by reducing the hole size
            # textinfo='label+percent',  # Display category labels and percentages
            # textposition='inside'  # Place labels inside the pie slices
        ), 
        row=row, 
        col=col
    )

# Update layout for the entire figure
fig.update_layout(
    title=dict(
        text="Normalized Contribution of Each Category by Artist",
        font=dict(size=20),
        y=0.98  # Adjust the title position to be higher, closer to the top of the figure
    ),
    height=300 * ((len(unique_artists) // 5) + 1),  # Adjust height to create more space between rows while reducing the number of rows
    width=1000,  # Increase the width of the entire figure to give more space for each subplot
    margin=dict(t=150, b=50),  # Increase top margin to prevent overlapping and create a gap between the title and subplots
    showlegend=False
)

# Reduce font size for subplot titles to reduce the chance of overlapping
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=12)  # Set font size of subplot titles

# Show the entire facet pie chart figure
fig.show()


In [ ]:
# Print all columns in the DataFrame
print(proportion_data.columns)

# Print first few rows to inspect data
print(proportion_data.head())


In [ ]:
import pandas as pd

# Group by 'song title' and 'Category' to get the count of stanzas for each category in each song
category_counts_song = proportion_data.groupby(['Song title', 'Category'])['Category Count'].sum().reset_index()

# Calculate the total number of categorized stanzas per song
total_categorized_stanzas_song = category_counts_song.groupby('Song title')['Category Count'].sum().reset_index(name='Total Categorized Stanzas')

# Merge the category counts with the total categorized stanzas for each song
proportion_data_song = pd.merge(category_counts_song, total_categorized_stanzas_song, on='Song title')

# Calculate the normalized proportion of stanzas for each category within each song (第一層標準化)
proportion_data_song['Proportion In Song'] = proportion_data_song['Category Count'] / proportion_data_song['Total Categorized Stanzas']

# # Display the DataFrame to verify
# print(proportion_data_song.head())

# Group by 'Category' to get the sum of all proportions across all songs for each category
category_total_proportion = proportion_data_song.groupby('Category')['Proportion In Song'].sum().reset_index(name='Category Total Proportion')

# Merge with the original DataFrame to add the total proportions for each category
proportion_data_final = pd.merge(proportion_data_song, category_total_proportion, on='Category')

# Calculate the final normalized contribution for each category in each song (第二層標準化)
proportion_data_final['Normalized Contribution'] = proportion_data_final['Proportion In Song'] / proportion_data_final['Category Total Proportion']

# Display the DataFrame to verify
print(proportion_data_final.head())

import plotly.express as px

# Create a grouped bar chart to visualize the final normalized contribution of each category by song
fig = px.bar(
    proportion_data_final,
    y='Song title',
    x='Normalized Contribution',
    color='Category',
    color_discrete_map= color_map,
    barmode='group',  # Use grouped bars to compare categories within each song
    title='Final Normalized Contribution of Each Category by Song',
    labels={
        'song title': 'Song Title',
        'Normalized Contribution': 'Final Normalized Contribution',
        'Category': 'Category'
    }
)

# Adjust layout for better spacing and readability
fig.update_layout(
    width=800,  # Set width for better readability
    height=1400,  # Set height to make sure all songs are easily viewable
    title_font_size=20,
    yaxis_tickangle=-45,  # Rotate song titles for better readability
    xaxis=dict(title='Normalized Contribution', range=[0, 0.15]),  # Set y-axis to range from 0 to 1
    margin=dict(t=80, b=100)
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd

# Group by 'artist' and 'Category' to get the count of stanzas for each category for each artist across all songs
category_counts_artist = proportion_data.groupby(['Artist', 'Category'])['Category Count'].sum().reset_index()

# Calculate the total number of categorized stanzas per artist across all their songs
total_categorized_stanzas_artist = category_counts_artist.groupby('Artist')['Category Count'].sum().reset_index(name='Total Categorized Stanzas')

# Merge the category counts with the total categorized stanzas for each artist
proportion_data_artist = pd.merge(category_counts_artist, total_categorized_stanzas_artist, on='Artist')

# Calculate the proportion of stanzas for each category within each artist (第一層標準化)
proportion_data_artist['Proportion In Artist'] = proportion_data_artist['Category Count'] / proportion_data_artist['Total Categorized Stanzas']

# # Display the DataFrame to verify
# print(proportion_data_artist.head())

# Group by 'Category' to get the sum of all proportions across all artists for each category
category_total_proportion_artist = proportion_data_artist.groupby('Category')['Proportion In Artist'].sum().reset_index(name='Category Total Proportion')

# Merge with the original DataFrame to add the total proportions for each category
proportion_data_final_artist = pd.merge(proportion_data_artist, category_total_proportion_artist, on='Category')

# Calculate the final normalized contribution for each category by artist (第二層標準化)
proportion_data_final_artist['Normalized Contribution'] = proportion_data_final_artist['Proportion In Artist'] / proportion_data_final_artist['Category Total Proportion']

# Display the DataFrame to verify
print(proportion_data_final_artist.head())

import plotly.express as px

# Create a grouped bar chart to visualize the final normalized contribution of each category by artist
fig = px.bar(
    proportion_data_final_artist,
    y='Artist',
    x='Normalized Contribution',
    color='Category',
    color_discrete_map= color_map,
    barmode='group',  # Use grouped bars to compare categories within each artist
    title='Final Normalized Contribution of Each Category by Artist',
    labels={
        'artist': 'Artist',
        'Normalized Contribution': 'Final Normalized Contribution',
        'Category': 'Category'
    }
)

# Adjust layout for better spacing and readability
fig.update_layout(
    width=600,  # Set width for better readability
    height=1200,  # Set height to make sure all artists are easily viewable
    title_font_size=20,
    yaxis_tickangle=-45,  # Rotate artist names for better readability
    xaxis=dict(title='Normalized Contribution', range=[0, 0.2]),  # Set y-axis to range from 0 to 1
    margin=dict(t=80, b=100)
)

# Show the plot
fig.show()


In [ ]:
df_melted = df.melt(
    id_vars=['Song title', 'Artist', 'Stanza Number',], 
    value_vars=['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness'],
    var_name='Category', value_name='Value'
)

# 只保留標籤為 1 的行（表示該段落屬於該分類）
df_melted = df_melted[df_melted['Value'] == 1]

# 將數據按歌曲和段落進行排序，方便後續的可視化
df_melted = df_melted.sort_values(by=['Song title', 'Stanza Number'])

# 查看整理後的數據
print(df_melted.head())

# # 將所有歌曲分為幾批，每次展示一批
# batch_size = 10
# total_batches = len(df_melted['Song title'].unique()) // batch_size + 1

# # 遍歷每個批次，創建單獨的圖表
# for i in range(total_batches):
#     songs_to_plot = df_melted['Song title'].unique()[i * batch_size:(i + 1) * batch_size]
#     filtered_data = df_melted[df_melted['Song title'].isin(songs_to_plot)]

#     fig = px.line(
#         filtered_data,
#         x='Stanza Number',
#         y='Value',
#         color='Category',
#         facet_row='Song title',
#         title=f'Category Emphasis by Stanza for Songs Batch {i + 1}',
#         labels={
#             'stanza number': 'Stanza Number',
#             'Value': 'Emphasis (Presence of Category)',
#             'Category': 'Category',
#             'song title': 'Song Title'
#         }
#     )

#     fig.update_layout(
#         width=1200,
#         height=3000,  # 為每批次設置合理的高度
#         title_font_size=20,
#         # facet_row_spacing=0.005,
#         xaxis=dict(title='Stanza Number'),
#         yaxis=dict(title='Emphasis (Presence of Category)', range=[0, 1]),
#         margin=dict(t=100, b=100),
#         showlegend=True
#     )

#     # 顯示該批次的圖表
#     fig.show()


In [ ]:
# import plotly.express as px

# # Visualize stanza-wise emphasis for each category using a stacked bar chart
# fig = px.bar(
#     stanza_category_pivot,
#     x='Stanza Number',
#     y=['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness'],
#     title='Category Emphasis by Stanza for Song',
#     labels={
#         'stanza number': 'Stanza Number',
#         'value': 'Count',
#         'variable': 'Category'
#     },
#     barmode='stack'
# )

# # Update layout for better visualization
# fig.update_layout(
#     width=1000,  # Set width for better readability
#     height=600,  # Set height to ensure all stanza numbers are visible
#     title_font_size=20,
#     xaxis=dict(title='Stanza Number'),
#     yaxis=dict(title='Category Emphasis'),
#     legend_title_text='Category'
# )

# # Show the plot
# fig.show()


In [ ]:
# import plotly.express as px

# 使用 facet_col 來對每首歌進行獨立的可視化，展示分類隨段落的變化
# fig = px.line(
#     df_melted,
#     x='Stanza Number',
#     y='Value',
#     color='Category',
#     color_discrete_map= color_map,
#     facet_col='Song title',
#     facet_col_wrap=5,  # 每行展示4個子圖，根據歌曲數量調整
#     title='Category Emphasis Through Time in Each Song',
#     labels={
#         'stanza number': 'Stanza Number',
#         'Value': 'Presence of Category',
#         'Category': 'Category',
#         'song title': 'Song Title'
#     }
# )

# fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# # 更新佈局以改善圖表的顯示效果
# fig.update_layout(
#     width=1000,  # 設置寬度，讓多個子圖能橫向顯示
#     height=2000,  # 增加高度來適應所有歌曲的顯示
#     title_font_size=20,
#     xaxis=dict(title='Stanza Number'),
#     yaxis=dict(title='Presence of Category', range=[0, 1]),
#     margin=dict(t=100, b=100),
#     showlegend=True  # 顯示圖例，方便理解不同分類
# )

# # 顯示圖表
# fig.show()


In [ ]:


# 獲取所有的歌曲名稱
unique_songs = df['Song title'].unique()

# 創建子圖，每行一首歌，每行有三個子圖（每個子圖對應一個分類）
fig = make_subplots(
    rows=len(unique_songs), cols=3,  # 每首歌一行，每行有三個子圖
    subplot_titles=[f"{song}" for song in unique_songs for category in ['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness']],
    vertical_spacing=0.01,  # 設置非常小的垂直間距，避免重疊問題
    horizontal_spacing=0.05  # 設置合理的水平間距
)

# 遍歷每首歌，並為每個分類創建子圖
for i, song in enumerate(unique_songs):
    # 為當前歌曲創建過濾的數據
    song_data = df[df['Song title'] == song]

    # 為每個分類創建子圖
    for j, category in enumerate(['Reinforce gender stereotype', 'Challenge gender stereotype', 'Self-awareness']):
        # 過濾出當前分類的數據
        category_data = song_data[['Stanza Number', category]].rename(columns={category: 'Value'})

        # 添加到子圖中
        fig.add_trace(
            go.Bar(x=category_data['Stanza Number'], y=category_data['Value'], name=f"{song}", marker_color=color_map[category]),
            row=i + 1, col=j + 1,
        )

# 更新佈局
fig.update_layout(
    height=6000,  # 根據歌曲數量調整總高度，設置為較大的值來適應更多行
    width=1000,  # 設置圖表的總寬度
    title_text='Category Emphasis Through Time for Each Song',
    showlegend=False  # 隱藏總體圖例
)

# 更新每個子圖的標題，只顯示歌曲名稱，並調整字體大小
fig.for_each_annotation(lambda a: a.update(text=a.text.split(" - ")[0], font=dict(size=10)))

# 顯示圖表
fig.show()


In [ ]:
# import plotly.express as px

# # 獲取所有的歌曲名稱
# unique_songs = df['Song title'].unique()

# # 遍歷每首歌，為每首歌創建一個分組條形圖
# for song in unique_songs:
#     # 過濾出當前歌曲的數據
#     song_data = df_melted[df_melted['Song title'] == song]

#     # 創建分組條形圖，顯示每個段落的不同分類
#     fig = px.bar(
#         song_data,
#         x='Stanza Number',
#         y='Value',
#         color='Category',
#         barmode='group',  # 使用分組條形圖模式
#         title=f'"{song}"',
#         labels={
#             'stanza number': 'Stanza Number',
#             'Value': 'Emphasis (Presence of Category)',
#             'Category': 'Category'
#         },
#         color_discrete_map=color_map  # 使用固定的顏色映射，保持一致
#     )

#     # 更新佈局
#     fig.update_layout(
#         width=800,
#         height=500,
#         title_font_size=18,
#         xaxis=dict(title='Stanza Number'),
#         yaxis=dict(title='Emphasis (Presence of Category)', range=[0, 1]),
#         margin=dict(t=50, b=50),
#         showlegend=True
#     )

#     # 顯示該首歌的圖表
#     fig.show()


In [ ]:
fig = px.bar(
    df_melted,
    x='Stanza Number',
    y='Value',
    color='Category',
    barmode='group',  # 分組條形圖模式
    facet_col='Song title',  # 每個子圖對應一首歌
    facet_col_wrap=4,  # 每行顯示 5 個子圖
    title='Category Emphasis Through Time for Each Song',
    labels={
        'Stanza Number': 'Stanza Number',
        'Value': 'Emphasis (Presence of Category)',
        'Category': 'Category'
    },
    color_discrete_map=color_map  # 使用固定的顏色映射，保持一致
)

# 更新佈局
fig.update_layout(
    width=1250,  # 根據子圖數量調整寬度
    height=3000,  # 根據行數調整高度
    title_font_size=20,
    xaxis=dict(title='Stanza Number'),
    yaxis=dict(title='Emphasis (Presence of Category)', range=[0, 1]),
    margin=dict(t=100, b=100),
    showlegend=True
)

# 更新每個子圖標題，只顯示歌曲名稱，並調整字體大小
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].strip(), font=dict(size=12)))

# 顯示圖表
fig.show()

In [ ]:
# 合併所有 stanza text 為一個字符串
all_text = ' '.join(df['Stanza Text'].astype(str))

# 創建詞雲
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='magma',  # 可以根據喜好選擇其他配色方案
    max_words=200,  # 控制顯示的最多單詞數量
    stopwords=set(['are', 'on', 'but', 'and', 'or', 'the', 'a', 'of', 'in', 'to', 'it', 'ooh', 'oh', 'yeah', 'what', 'can', 'wanna', 'til', 'beep', 'just', 'toot', 'hey', 'let', 'ya', 'that', 'hmm'])  # 可自行添加常見的停用詞
).generate(all_text)

# 顯示詞雲
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear') 
plt.axis('off')  # 隱藏坐標軸
plt.title('WordCloud of All Song', fontsize=16)
plt.show()

In [ ]:
from PIL import Image

# 合併所有 stanza text 為一個字符串
all_text = ' '.join(df['Stanza Text'].astype(str))

# 讀取你創建的包含 "Disco" 文字的形狀圖片
disco_mask = np.array(Image.open("disco_layout.png"))

# 創建詞雲
wordcloud = WordCloud(
    width=1000,
    height=600,
    background_color='black',  # 設置背景顏色為黑色，與蒙版背景一致
    mask=disco_mask,  # 使用蒙版來定義詞雲形狀
    contour_color='white',  # 為詞雲形狀添加白色輪廓
    contour_width=3,  # 輪廓寬度設置為 3 來突出顯示形狀
    colormap='plasma',  # 可以根據喜好選擇其他顏色映射
    max_words=1000,  # 增加最大單詞數量，以填滿更多空間
    max_font_size=400,  # 增大最大字體大小
    min_font_size=20,  # 設置最小字體大小
    prefer_horizontal=0.1,  # 減少水平排列的概率，讓單詞更隨意排列
    random_state=42,
    stopwords=set(['are', 'on', 'but', 'and', 'or', 'the', 'a', 'of', 'in', 'to', 'it', 'ooh', 'oh', 'yeah', 'what', 'can', 'wanna', 'til', 'beep', 'just', 'toot', 'hey', 'let', 'ya', 'that', 'hmm'])  # 可自行添加常見的停用詞
).generate(all_text)

# 顯示詞雲
plt.figure(figsize=(10, 4))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')  # 隱藏坐標軸
plt.title('WordCloud of All Song', fontsize=16)
plt.show()

In [ ]:
unique_songs = df['Song title'].unique()

# 遍歷每首歌並生成詞雲
for song in unique_songs:
    # 過濾出當前歌曲的所有 stanza text
    song_data = df[df['Song title'] == song]
    song_text = ' '.join(song_data['Stanza Text'].astype(str))

    # 創建詞雲
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='inferno',  # 你可以替換為任何顏色映射，例如 'viridis', 'inferno', etc.
        max_words=50,
        stopwords=set(['uh', 'so', 'ah', 'whoa', 'are', 'on', 'but', 'and', 'or', 'the', 'a', 'of', 'in', 'to', 'it', 'ooh', 'oh', 'yeah', 'what', 'can', 'wanna', 'til', 'beep', 'just', 'toot', 'hey', 'let', 'ya', 'that', 'hmm', 'Did', 'did', 'woo', 'huh', 'as', 'by', 'be', 'hoo', 'Oooh', 'ey', 'whoah', 'Whoah'])  # 可自行添加常見的停用詞
).generate(song_text)

    # 顯示詞雲
    plt.figure(figsize=(8, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  # 隱藏坐標軸
    plt.title(f'WordCloud for "{song}"', fontsize=16)
    plt.show()
